# Future AQI Prediction
 The selected hyperparameters come from the 2024 validation search; the 2025 test set is used only for final evaluation.


In [ ]:
"""
Future AQI Prediction — Final Training and Export Pipeline
---------------------------------------------------------
Input:  ./Datasets/Final_Dataset.csv
Outputs:
  saved_models/       Four trained XGBoost models (.json)
  predictions/        Horizon-specific 2025 predictions
  powerbi_data/       Combined wide and long Power BI datasets

Run from the project root. The script uses the selected hyperparameters from
validation tuning; it does not retune or use the 2025 test set for model selection.
"""

from pathlib import Path
import json
import numpy as np
import pandas as pd

from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from xgboost import XGBRegressor


# ---------------------------------------------------------------------
# 1. Configuration
# ---------------------------------------------------------------------

INPUT_FILE = Path("./Datasets/Final_Dataset.csv")
OUTPUT_DIR = Path(".")
RANDOM_STATE = 42

# Selected using 2024 validation results
BEST_PARAMS = {
    "1h_A": {
        "n_estimators": 700, "max_depth": 6, "learning_rate": 0.03,
        "min_child_weight": 5, "subsample": 0.8, "colsample_bytree": 1.0,
    },
    "1h_B": {
        "n_estimators": 700, "max_depth": 6, "learning_rate": 0.03,
        "min_child_weight": 5, "subsample": 0.8, "colsample_bytree": 1.0,
    },
    "24h_A": {
        "n_estimators": 300, "max_depth": 6, "learning_rate": 0.03,
        "min_child_weight": 1, "subsample": 0.8, "colsample_bytree": 1.0,
    },
    "24h_B": {
        "n_estimators": 700, "max_depth": 6, "learning_rate": 0.03,
        "min_child_weight": 5, "subsample": 0.8, "colsample_bytree": 1.0,
    },
}

WEATHER_COLS = [
    "temperature_2m (°C)",
    "relative_humidity_2m (%)",
    "dew_point_2m (°C)",
    "precipitation (mm)",
    "surface_pressure (hPa)",
    "wind_speed_10m (km/h)",
    "wind_direction_10m (°)",
    "shortwave_radiation (W/m²)",
    "cloud_cover (%)",
]
LOCATION_COLS = ["elevation", "latitude", "longitude"]
# Add your exact station identifier column here if it has a different name.
STATION_CANDIDATES = ["Station", "Station_Name", "station", "station_name", "Station Name"]
TARGET_COLS = ["AQI_future_1", "AQI_future_24"]
EXCLUDED_FEATURES = ["Time", "_Timestamp", *TARGET_COLS]


# ---------------------------------------------------------------------
# 2. Load data and prepare timestamp/calendar features
# ---------------------------------------------------------------------

if not INPUT_FILE.exists():
    raise FileNotFoundError(f"Dataset not found: {INPUT_FILE.resolve()}")

df = pd.read_csv(INPUT_FILE)

required_columns = {
    "Day", "Month", "Year", "Time", "Day_of_Week",
    *TARGET_COLS, *WEATHER_COLS, *LOCATION_COLS,
}
missing_required = sorted(required_columns.difference(df.columns))
if missing_required:
    raise ValueError(f"Missing required dataset columns: {missing_required}")

df["_Timestamp"] = pd.to_datetime(
    df["Day"].astype(str).str.zfill(2) + "-"
    + df["Month"].astype(str).str.zfill(2) + "-"
    + df["Year"].astype(str) + " "
    + df["Time"].astype(str),
    format="%d-%m-%Y %H:%M:%S",
    errors="raise",
)

# Keep Hour numeric; do not use the original Time string as a model feature.
df["Hour"] = df["_Timestamp"].dt.hour

# Encode categorical calendar columns only if they have not already been encoded.
if "Season" in df.columns:
    df = pd.get_dummies(df, columns=["Season"], prefix="Season", dtype=int)

if "Day_of_Week" in df.columns:
    df = pd.get_dummies(df, columns=["Day_of_Week"], prefix="Day", dtype=int)

print(f"Loaded dataset: {df.shape[0]:,} rows × {df.shape[1]} columns")


# ---------------------------------------------------------------------
# 3. Define feature sets
# ---------------------------------------------------------------------

# All non-weather predictors, excluding timestamp helpers and future targets.
features_A = [
    col for col in df.columns
    if col not in set(WEATHER_COLS + EXCLUDED_FEATURES)
]
# Model B adds the nine weather predictors.
features_B = features_A + WEATHER_COLS

# Validate feature lists before training.
for model_name, feature_cols in [("Model A", features_A), ("Model B", features_B)]:
    missing = [col for col in feature_cols if col not in df.columns]
    non_numeric = df[feature_cols].select_dtypes(exclude="number").columns.tolist()
    leakage = [
        col for col in feature_cols
        if col in TARGET_COLS or col == "_Timestamp" or col == "Time"
    ]
    if missing or non_numeric or leakage:
        raise ValueError(
            f"{model_name} feature validation failed: "
            f"missing={missing}, non_numeric={non_numeric}, leakage={leakage}"
        )
    print(f"{model_name}: {len(feature_cols)} numeric features")

print("Hour included:", "Hour" in features_A)
print("Weather feature count:", len(WEATHER_COLS))


# ---------------------------------------------------------------------
# 4. Chronological train/validation/test split
# ---------------------------------------------------------------------

def split_by_year(data: pd.DataFrame, target_col: str, horizon_hours: int):
    """Split by input year and ensure each target remains in the same year block."""
    subset = data.dropna(subset=[target_col]).copy()
    target_time = subset["_Timestamp"] + pd.Timedelta(hours=horizon_hours)

    train_mask = (
        subset["_Timestamp"].dt.year.between(2020, 2023)
        & (target_time < pd.Timestamp("2024-01-01"))
    )
    val_mask = (
        (subset["_Timestamp"].dt.year == 2024)
        & (target_time < pd.Timestamp("2025-01-01"))
    )
    test_mask = (
        (subset["_Timestamp"].dt.year == 2025)
        & (target_time < pd.Timestamp("2026-01-01"))
    )

    return (
        subset.loc[train_mask].copy(),
        subset.loc[val_mask].copy(),
        subset.loc[test_mask].copy(),
    )


splits = {
    "1h": split_by_year(df, "AQI_future_1", 1),
    "24h": split_by_year(df, "AQI_future_24", 24),
}

for horizon, (train_df, val_df, test_df) in splits.items():
    print(
        f"{horizon}: train={len(train_df):,}, "
        f"validation={len(val_df):,}, test={len(test_df):,}"
    )


# ---------------------------------------------------------------------
# 5. Build feature matrices and target vectors
# ---------------------------------------------------------------------

def make_xy(split_df: pd.DataFrame, feature_cols: list[str], target_col: str):
    X = split_df.loc[:, feature_cols]
    y = split_df[target_col]
    return X, y


data = {}
for horizon, (train_df, val_df, test_df) in splits.items():
    target_col = "AQI_future_1" if horizon == "1h" else "AQI_future_24"
    for model_key, feature_cols in [("A", features_A), ("B", features_B)]:
        X_train, y_train = make_xy(train_df, feature_cols, target_col)
        X_val, y_val = make_xy(val_df, feature_cols, target_col)
        X_test, y_test = make_xy(test_df, feature_cols, target_col)

        data[f"{horizon}_{model_key}"] = {
            "X_train": X_train, "y_train": y_train,
            "X_val": X_val, "y_val": y_val,
            "X_test": X_test, "y_test": y_test,
            "test_df": test_df,
            "features": feature_cols,
        }


# ---------------------------------------------------------------------
# 6. Final training on train + validation; evaluate once on 2025 test
# ---------------------------------------------------------------------

def regression_metrics(y_true, y_pred):
    return {
        "MAE": mean_absolute_error(y_true, y_pred),
        "RMSE": np.sqrt(mean_squared_error(y_true, y_pred)),
        "R2": r2_score(y_true, y_pred),
    }


def make_model(params: dict):
    return XGBRegressor(
        objective="reg:squarederror",
        tree_method="hist",
        device="cuda",       # Change to "cpu" if CUDA is unavailable.
        n_jobs=-1,
        random_state=RANDOM_STATE,
        **params,
    )


models = {}
metrics_rows = []
predictions_by_horizon = {}

for model_key in ["1h_A", "1h_B", "24h_A", "24h_B"]:
    item = data[model_key]

    # Retrain on train + validation after hyperparameters were selected.
    X_train_final = pd.concat([item["X_train"], item["X_val"]], axis=0)
    y_train_final = pd.concat([item["y_train"], item["y_val"]], axis=0)

    model = make_model(BEST_PARAMS[model_key])
    model.fit(X_train_final, y_train_final, verbose=False)

    y_pred = model.predict(item["X_test"])
    scores = regression_metrics(item["y_test"], y_pred)

    models[model_key] = model
    metrics_rows.append({
        "Forecast_Horizon": "1 Hour" if model_key.startswith("1h") else "24 Hours",
        "Model": "Model A (No Weather)" if model_key.endswith("_A")
                 else "Model B (With Weather)",
        **scores,
    })

    horizon = "1h" if model_key.startswith("1h") else "24h"
    model_label = "A" if model_key.endswith("_A") else "B"
    predictions_by_horizon.setdefault(horizon, {})
    predictions_by_horizon[horizon][model_label] = y_pred

metrics_df = pd.DataFrame(metrics_rows)
print("\nFinal 2025 test metrics:")
print(metrics_df.to_string(index=False, float_format=lambda x: f"{x:.4f}"))


# ---------------------------------------------------------------------
# 7. Save trained models and test predictions
# ---------------------------------------------------------------------

for folder in ["saved_models", "predictions", "powerbi_data"]:
    (OUTPUT_DIR / folder).mkdir(parents=True, exist_ok=True)

model_filenames = {
    "1h_A": "xgb_1h_no_weather.json",
    "1h_B": "xgb_1h_with_weather.json",
    "24h_A": "xgb_24h_no_weather.json",
    "24h_B": "xgb_24h_with_weather.json",
}

for key, model in models.items():
    model_path = OUTPUT_DIR / "saved_models" / model_filenames[key]
    model.save_model(model_path)
    print(f"Saved model: {model_path}")

prediction_frames = {}

for horizon in ["1h", "24h"]:
    split_df = splits[horizon][2]
    target_col = "AQI_future_1" if horizon == "1h" else "AQI_future_24"

    # Arrays are assigned positionally to preserve row alignment with the test split.
    pred_df = pd.DataFrame({
        "Timestamp": split_df["_Timestamp"].to_numpy(),
        "AQI_actual": split_df[target_col].to_numpy(),
        "AQI_pred_A_no_weather": predictions_by_horizon[horizon]["A"],
        "AQI_pred_B_with_weather": predictions_by_horizon[horizon]["B"],
    })

    # Preserve station identifiers in the export when present in the source data.
    station_cols = [col for col in STATION_CANDIDATES if col in split_df.columns]
    for col in station_cols:
        pred_df[col] = split_df[col].to_numpy()

    pred_df = pred_df.sort_values("Timestamp").reset_index(drop=True)
    out_path = OUTPUT_DIR / "predictions" / f"predictions_2025_{horizon}.csv"
    pred_df.to_csv(out_path, index=False)
    prediction_frames[horizon] = pred_df
    print(f"Saved predictions: {out_path} ({len(pred_df):,} rows)")


# ---------------------------------------------------------------------
# 8. Prepare Power BI wide and long datasets
# ---------------------------------------------------------------------

wide_frames = []
for horizon, pred_df in prediction_frames.items():
    frame = pred_df.rename(columns={
        "AQI_pred_A_no_weather": "Model_A_No_Weather",
        "AQI_pred_B_with_weather": "Model_B_With_Weather",
    }).copy()
    frame["Forecast_Horizon"] = "1 Hour" if horizon == "1h" else "24 Hours"
    wide_frames.append(frame)

powerbi_wide = pd.concat(wide_frames, ignore_index=True)
powerbi_wide["Timestamp"] = pd.to_datetime(powerbi_wide["Timestamp"])
powerbi_wide["Date"] = powerbi_wide["Timestamp"].dt.date
powerbi_wide["Year"] = powerbi_wide["Timestamp"].dt.year
powerbi_wide["Month"] = powerbi_wide["Timestamp"].dt.month
powerbi_wide["Month_Name"] = powerbi_wide["Timestamp"].dt.strftime("%b")

wide_path = OUTPUT_DIR / "powerbi_data" / "AQI_Predictions_2025_PowerBI.csv"
powerbi_wide.to_csv(wide_path, index=False)

powerbi_id_cols = [
    "Timestamp", "Date", "Year", "Month", "Month_Name",
    "Forecast_Horizon", "AQI_actual",
]
powerbi_id_cols += [col for col in STATION_CANDIDATES if col in powerbi_wide.columns]

powerbi_long = powerbi_wide.melt(
    id_vars=powerbi_id_cols,
    value_vars=["Model_A_No_Weather", "Model_B_With_Weather"],
    var_name="Model",
    value_name="AQI_predicted",
)
powerbi_long["Absolute_Error"] = (
    powerbi_long["AQI_actual"] - powerbi_long["AQI_predicted"]
).abs()
powerbi_long["Residual"] = (
    powerbi_long["AQI_actual"] - powerbi_long["AQI_predicted"]
)

long_path = OUTPUT_DIR / "powerbi_data" / "AQI_Predictions_PowerBI_Long.csv"
powerbi_long.to_csv(long_path, index=False)

metrics_path = OUTPUT_DIR / "powerbi_data" / "AQI_Test_Metrics_2025.csv"
metrics_df.to_csv(metrics_path, index=False)

print("\nPower BI files saved:")
print(f"- {wide_path}")
print(f"- {long_path}")
print(f"- {metrics_path}")
